# Classic RAG Implementation

This notebook demonstrates a complete RAG pipeline using LangChain + Ollama (local, free).

## What You'll Learn

1. **Document Loading** - How to load external knowledge into your system
2. **Text Chunking** - Why and how to split documents into manageable pieces
3. **Embedding Creation** - Converting text into numerical representations
4. **Vector Storage** - Storing embeddings for fast similarity search
5. **Retrieval** - Finding relevant documents based on semantic similarity
6. **Answer Generation** - Using the LLM to generate answers with retrieved context

## Why RAG?

Large Language Models (LLMs) have three key limitations that RAG addresses:

| Limitation | Description | How RAG Helps |
|-----------|------------|---------------|
| **Hallucinations** | LLMs can generate false information | Ground responses in real documents |
| **Knowledge Cutoff** | Training data becomes outdated | Retrieve up-to-date information |
| **No Source Attribution** | Can't verify information source | Provide source citations |

## Architecture Overview

```
┌─────────────┐    ┌─────────────┐    ┌─────────────┐
│   Document  │───▶│    Chunk    │───▶│  Embedding  │
│   Loading   │    │   Splitter  │    │   Model     │
└─────────────┘    └─────────────┘    └──────┬──────┘
                                             │
                                             ▼
┌─────────────┐    ┌─────────────┐    ┌─────────────┐
│   Final     │◀───│     LLM     │◀───│   Retrieved │
│   Answer    │    │  Generation │    │   Context   │
└─────────────┘    └─────────────┘    └──────┬──────┘
                                             │
                                             ▼
                                       ┌─────────────┐
                                       │   Vector    │
                                       │   Store     │
                                       └─────────────┘
```

## Prerequisites

Install Ollama and pull the required models:
```bash
ollama pull llama3.2
ollama pull nomic-embed-text
```

In [ ]:
# Install dependencies
!pip install langchain langchain-community langchain-ollama chromadb sentence-transformers

In [50]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate


## Step 1: Load Documents

**What:** Loading raw documents from various sources (files, databases, websites).

**Why:** Your knowledge base needs to be loaded into the system before it can be used for retrieval. This example uses a simple text file, but LangChain supports many loaders (PDF, CSV, websites, Notion, etc.).

In [51]:
# Create sample document
sample_text = """
Retrieval-Augmented Generation (RAG) is an AI framework that enhances 
Large Language Models by retrieving relevant information from external 
knowledge bases. RAG addresses key limitations of LLMs including 
hallucinations, knowledge cutoff, and lack of source attribution.

The basic RAG pipeline consists of three components:
1. Retrieval: Finding relevant documents
2. Augmentation: Adding context to the prompt
3. Generation: Producing the final response

RAG is particularly useful for:
- Question answering systems
- Knowledge base chatbots
- Enterprise search
- Customer support automation
"""

# Save sample document
with open("sample_doc.txt", "w") as f:
    f.write(sample_text)

# Load document
loader = TextLoader("sample_doc.txt")
documents = loader.load()

print(f"Loaded {len(documents)} document(s)")

Loaded 1 document(s)


## Step 2: Split Documents

**What:** Splitting large documents into smaller chunks.

**Why:** 
- LLMs have context windows (limits on input size)
- Smaller chunks = more precise retrieval
- If chunks are too large, relevant info gets diluted
- If chunks are too small, context may be lost

## Key Parameters Explained

| Parameter | Value | Why |
|-----------|-------|-----|
| `chunk_size` | 200 | Small enough for precise retrieval, large enough for context |
| `chunk_overlap` | 50 | Ensures context isn't lost at chunk boundaries |

> **Tip:** In production, experiment with chunk sizes (256-512) based on your use case. Technical docs often work better with larger chunks than conversational data.

In [52]:
# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

texts = text_splitter.split_documents(documents)

print(f"Created {len(texts)} chunks")
for i, text in enumerate(texts[:3]):
    print(f"\nChunk {i+1}: {text.page_content[:100]}...")

Created 4 chunks

Chunk 1: Retrieval-Augmented Generation (RAG) is an AI framework that enhances 
Large Language Models by retr...

Chunk 2: knowledge bases. RAG addresses key limitations of LLMs including 
hallucinations, knowledge cutoff, ...

Chunk 3: The basic RAG pipeline consists of three components:
1. Retrieval: Finding relevant documents
2. Aug...


## Step 3: Create Embeddings & Vector Store

**What:** Converting text chunks into numerical vectors (embeddings) and storing them.

**Why:** 
- Embeddings capture **semantic meaning**, not just exact matches
- Similar concepts have similar vectors (near each other in vector space)
- This enables "search by meaning" rather than keyword matching

## How Retrieval Works

```
Query: "What is RAG?"
       │
       ▼
┌──────────────────┐
│  Embed Query     │ ──▶ [0.12, -0.34, 0.56, ...]
└──────────────────┘
       │
       ▼
┌──────────────────┐
│  Cosine         │ ──▶ Find nearest vectors
│  Similarity     │
└──────────────────┘
       │
       ▼
Most similar chunks returned!
```

> **Note:** We use `nomic-embed-text` - an open-source embedding model that works locally with Ollama.

In [53]:
# Create embeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Create vector store
vectorstore = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Vector store created successfully!")

Vector store created successfully!


## Step 4: Create QA Chain

In [54]:
# Create LLM
llm = ChatOllama(model="llama3.2", temperature=0)

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

# Create QA chain using LCEL
prompt = PromptTemplate.from_template("Use the following context to answer the question. If you cannot find the answer in the context, say you don't know.\n\nContext: {context}\nQuestion: {question}\nAnswer:")

qa_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("QA chain ready!")

QA chain ready!


## Step 5: Query the System

In [55]:
# Ask a question
query = "What are the three components of RAG?"

result = qa_chain.invoke(query)

print("Question:", query)
print("\nAnswer:", result)
print("\nSource Documents:")
# Note: LCEL chain returns directly, use retriever.invoke(query) for source docs
print(f"- {doc.page_content[:80]}...")

Question: What are the three components of RAG?

Answer: The three components of RAG are:

1. Retrieval: Finding relevant documents
2. Augmentation: Adding context to the prompt
3. Generation: Producing the final response

Source Documents:
- The basic RAG pipeline consists of three components:
1. Retrieval: Finding relev...


## Try More Queries

In [56]:
# More example queries
queries = [
    "What is RAG used for?",
    "How does RAG reduce hallucinations?",
    "What are the limitations of LLMs that RAG addresses?"
]

for q in queries:
    result = qa_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {result}\n")

Q: What is RAG used for?
A: RAG is particularly useful for: 
- Question answering systems
- Knowledge base chatbots
- Enterprise search
- Customer support automation

Q: How does RAG reduce hallucinations?
A: I don't know. The context only mentions that RAG addresses hallucinations as one of its key limitations of LLMs, but it doesn't provide information on how RAG specifically reduces hallucinations.

Q: What are the limitations of LLMs that RAG addresses?
A: hallucinations, knowledge cutoff, and lack of source attribution.



## Next Steps

- Try with real documents
- Experiment with different chunk sizes
- Add query optimization techniques
- Explore different retrieval strategies